In [1]:
# CELL 20 — Full build setup: paths, chunk plan, and the v2 generator (identical to CELL 9; only paths, split rule and job list changed)
!pip install -q pikepdf pymupdf pypdf pebble

import os, io, re, glob, json, zlib, shutil, random, logging
import pandas as pd
import pikepdf, pypdf
import pymupdf as fitz
from pypdf import PdfWriter
from pypdf.generic import ArrayObject, StreamObject, NameObject
from PIL import Image, ImageDraw, ImageFont
from pebble import ProcessPool
from concurrent.futures import TimeoutError, as_completed
from tqdm.auto import tqdm

logging.getLogger("pypdf").setLevel(logging.ERROR)
logging.getLogger("pikepdf").setLevel(logging.ERROR)

# ---------- paths ----------
hits = sorted(glob.glob("/kaggle/input/**/clean_pool.csv", recursive=True))
assert hits, "clean_pool.csv not found - is PDP BPTP attached as input?"
CLEAN_POOL = hits[0]
SRC = os.path.join(os.path.dirname(CLEAN_POOL), "govdocs_pdfs")
assert os.path.isdir(SRC), f"govdocs_pdfs not found next to {CLEAN_POOL}"
WORK = "/kaggle/working/full"
OUT = f"{WORK}/tmp_pdfs"      # PDFs of the current chunk only (deleted after extraction)
META = f"{WORK}/meta"         # one small JSON per original (manifest rows)
CHUNKS = f"{WORK}/chunks"     # per-chunk results: features CSV + graphs file
for d in (OUT, META, CHUNKS):
    os.makedirs(d, exist_ok=True)

TIME_LIMIT = 180
CHUNK_SIZE = 300
WORKERS = 4
FAMILIES = ["acrobat distiller windows", "acrobat distiller for", "acrobat pdfwriter for",
            "adobe pdf library", "corel pdf engine"]
TOOLS = ["pikepdf", "pymupdf", "pypdf"]

LIT = re.compile(rb"\((?:[^()\\]|\\.)*\)", re.S)
TAIL = re.compile(rb"(?:\s+|-?(?:\d+\.?\d*|\.\d+)(?=[\s(\]])|\((?:[^()\\]|\\.)*\))*\s*(?:\]\s*TJ|Tj|'|\")")
VALUE = re.compile(rb"\d[\d,\.]*\d")

def make_stamp(text):
    img = Image.new("RGBA", (420, 140), (255, 255, 255, 0))
    dr = ImageDraw.Draw(img)
    dr.rectangle([4, 4, 415, 135], outline=(200, 0, 0, 255), width=8)
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", 60)
    except Exception:
        font = ImageFont.load_default()
    dr.text((24, 34), text, fill=(200, 0, 0, 255), font=font)
    buf = io.BytesIO()
    img.save(buf, "PNG")
    return buf.getvalue()

PNG_HONEST = make_stamp("RECEIVED")
PNG_FAKE = make_stamp("PAID")

def digit_positions(seg):
    pos, i, n = [], 1, len(seg) - 1
    while i < n:
        b = seg[i]
        if b == 0x5C:
            j = i + 1
            if j < n and 48 <= seg[j] <= 55:
                k = j
                while k < n and k < j + 3 and 48 <= seg[k] <= 55:
                    k += 1
                i = k
            else:
                i = j + 1
            continue
        if 48 <= b <= 57:
            pos.append(i)
        i += 1
    return pos

def candidates(data):
    digits, values = [], []
    for m in LIT.finditer(data):
        if not TAIL.match(data[m.end():m.end() + 300]):
            continue
        seg = m.group()
        digits += [m.start() + p for p in digit_positions(seg)]
        for v in VALUE.finditer(seg):
            if v.start() > 0 and seg[v.start() - 1] == 0x5C:
                continue
            if sum(c in b"0123456789" for c in v.group()) >= 3:
                values.append((m.start() + v.start(), m.start() + v.end()))
    return digits, values

def new_value(old, rng):
    s = old.decode("latin-1")
    two_dec = s.count(".") == 1 and len(s.split(".")[1]) == 2
    intpart = s.split(".")[0] if two_dec else s
    nd = max(1, sum(c.isdigit() for c in intpart) + rng.choice([-1, 1, 2]))
    n = rng.randint(10 ** (nd - 1), 10 ** nd - 1)
    out = (f"{n:,}" if "," in s else str(n)) + (f".{rng.randint(0, 99):02d}" if two_dec else "")
    if out == s:
        out = str(n + 1)
    return out.encode()

def pk_streams(pdf, i):
    c = pdf.pages[i].obj.get("/Contents")
    if c is None:
        return []
    return [s for s in (list(c) if isinstance(c, pikepdf.Array) else [c]) if isinstance(s, pikepdf.Stream)]

def choose_specs(src, rng):
    dspec = vspec = None
    with pikepdf.open(src) as p:
        order = list(range(len(p.pages)))
        rng.shuffle(order)
        for i in order:
            for k, s in enumerate(pk_streams(p, i)):
                try:
                    data = s.read_bytes()
                except Exception:
                    continue
                d, v = candidates(data)
                if dspec is None and d:
                    pos = rng.choice(d)
                    old = data[pos:pos + 1]
                    new = bytes([rng.choice([c for c in b"0123456789" if c != old[0]])])
                    dspec = dict(page=i, k=k, start=pos, end=pos + 1, old=old, new=new)
                if vspec is None and v:
                    st, en = rng.choice(v)
                    vspec = dict(page=i, k=k, start=st, end=en, old=data[st:en], new=new_value(data[st:en], rng))
                if dspec and vspec:
                    return dspec, vspec
    return dspec, vspec

def apply_spec(data, spec):
    if data[spec["start"]:spec["end"]] != spec["old"]:
        raise RuntimeError("edit location mismatch")
    return data[:spec["start"]] + spec["new"] + data[spec["end"]:]

def edit_pikepdf(pdf, spec):
    s = pk_streams(pdf, spec["page"])[spec["k"]]
    s.write(apply_spec(s.read_bytes(), spec))

def edit_fitz(doc, spec):
    x = doc[spec["page"]].get_contents()[spec["k"]]
    doc.update_stream(x, apply_spec(doc.xref_stream(x), spec))

def py_streams(page):
    c = page.get("/Contents")
    if c is None:
        return []
    c = c.get_object()
    out = []
    for it in (list(c) if isinstance(c, ArrayObject) else [c]):
        o = it.get_object()
        if isinstance(o, StreamObject):
            out.append(o)
    return out

def edit_pypdf(w, spec):
    o = py_streams(w.pages[spec["page"]])[spec["k"]]
    new = apply_spec(o.get_data(), spec)
    flt = o.get("/Filter")
    if flt is not None and flt != "/FlateDecode" and list(flt) != ["/FlateDecode"]:
        o[NameObject("/Filter")] = NameObject("/FlateDecode")
    if "/DecodeParms" in o:
        del o["/DecodeParms"]
    o.set_data(new)

def run_tool(tool, src, out, spec, mode):
    if tool == "pikepdf":
        with pikepdf.open(src) as p:
            if spec:
                edit_pikepdf(p, spec)
            if mode == "recompress":
                p.save(out, compress_streams=True, recompress_flate=True,
                       object_stream_mode=pikepdf.ObjectStreamMode.generate)
            elif mode == "linearize":
                p.save(out, linearize=True)
            else:
                p.save(out)
    elif tool == "pymupdf":
        d = fitz.open(src)
        if spec:
            edit_fitz(d, spec)
        if mode == "recompress":
            for pg in d:
                for x in pg.get_contents():
                    d.update_stream(x, d.xref_stream(x))
            d.save(out, garbage=4, deflate=True)
        else:
            d.save(out)
        d.close()
    else:
        w = PdfWriter(clone_from=src)
        if spec:
            edit_pypdf(w, spec)
        if mode == "recompress":
            for pg in w.pages:
                pg.compress_content_streams()
        with open(out, "wb") as f:
            w.write(f)

def fitz_incr(src, out, action):
    shutil.copy(src, out)
    d = fitz.open(out)
    try:
        if d.is_repaired:
            raise RuntimeError("file needed repair")
        res = action(d)
        d.saveIncr()
        return res
    finally:
        d.close()

def fitz_full(src, out, action):
    d = fitz.open(src)
    try:
        res = action(d)
        d.save(out)
        return res
    finally:
        d.close()

def add_note(d, i):
    r = d[i].rect
    d[i].add_text_annot(fitz.Point(r.x0 + 36, r.y0 + 36), "Reviewed")

def add_stamp(d, i, png):
    r = d[i].rect
    d[i].insert_image(fitz.Rect(r.x1 - 210, r.y0 + 40, r.x1 - 40, r.y0 + 97), stream=png, overlay=True)

def add_textbox(d, i, text):
    r = d[i].rect
    d[i].add_freetext_annot(fitz.Rect(r.x0 + 40, r.y1 - 80, r.x0 + 300, r.y1 - 58), text, fontsize=11)

def cover_replace(d, rng):
    order = list(range(len(d)))
    rng.shuffle(order)
    for i in order:
        words = [w for w in d[i].get_text("words") if re.search(r"\d", w[4]) and len(w[4]) >= 2]
        if not words:
            continue
        x0, y0, x1, y1, txt = rng.choice(words)[:5]
        r = fitz.Rect(x0, y0, x1, y1)
        if r.is_empty or r.height < 3:
            continue
        new = "".join(rng.choice("0123456789") if c.isdigit() else c for c in txt)
        if new == txt:
            new = txt[:-1] + ("7" if txt[-1] != "7" else "3")
        d[i].draw_rect(r, color=None, fill=(1, 1, 1), overlay=True)
        d[i].insert_text(fitz.Point(x0, y1 - r.height * 0.2), new, fontsize=max(5, r.height * 0.8),
                         fontname="helv", color=(0, 0, 0))
        return {"page": i, "old": txt, "new": new}
    raise RuntimeError("no numeric word for cover-and-replace")

def cleanup(stem, keep_meta=False):
    for p in glob.glob(os.path.join(OUT, f"{stem}__*.pdf")):
        os.remove(p)
    mp = os.path.join(META, stem + ".json")
    if not keep_meta and os.path.exists(mp):
        os.remove(mp)

def split_of(name):
    return "dev" if zlib.crc32(name.encode()) % 100 < 15 else "train"

# ---------- one original -> 15 versions (same as CELL 9; only the split line differs) ----------
def process(job):
    name, fam = job
    fitz.TOOLS.mupdf_display_errors(False)
    stem = name[:-4]
    src = os.path.join(SRC, name)
    meta_path = os.path.join(META, stem + ".json")
    if os.path.exists(meta_path):
        with open(meta_path) as fh:
            rows = json.load(fh)
        if all(os.path.exists(os.path.join(OUT, r["out_file"])) for r in rows):
            return name, rows, None
    cleanup(stem)
    try:
        rng = random.Random(zlib.crc32(name.encode()) + 2026)
        split = split_of(name)
        t_main, t_comp = rng.choice(TOOLS), rng.choice(TOOLS)
        dspec, vspec = choose_specs(src, rng)
        if dspec is None or vspec is None:
            raise RuntimeError("no editable number")
        P = lambda v: os.path.join(OUT, f"{stem}__{v}.pdf")
        pg = dspec["page"]

        shutil.copy(src, P("G0_original"))
        run_tool(t_main, src, P("G1_resave"), None, "resave")
        run_tool(t_comp, src, P("G2_recompress"), None, "recompress")
        fitz_incr(src, P("G3_note_incr"), lambda d: add_note(d, pg))
        run_tool("pikepdf", src, P("G4_linearize"), None, "linearize")
        fitz_full(src, P("G5_stamp_honest"), lambda d: add_stamp(d, pg, PNG_HONEST))
        fitz_incr(src, P("G6_textbox_honest"), lambda d: add_textbox(d, pg, "Approved - HR dept."))

        run_tool(t_main, src, P("F1_digit_resave"), dspec, "resave")
        run_tool(t_comp, src, P("F2_digit_recompress"), dspec, "recompress")
        fitz_incr(src, P("F3_digit_incr"), lambda d: edit_fitz(d, dspec))
        run_tool(t_main, src, P("F4_value_resave"), vspec, "resave")
        cov = fitz_full(src, P("F5_cover_replace"), lambda d: cover_replace(d, rng))
        run_tool("pikepdf", src, P("F6_digit_linearize"), dspec, "linearize")
        fitz_full(src, P("F7_stamp_fake"), lambda d: add_stamp(d, pg, PNG_FAKE))
        fitz_incr(src, P("F8_textbox_fake"), lambda d: add_textbox(d, pg, "TOTAL DUE: $98,450.00"))

        dold, dnew = dspec["old"].decode("latin-1"), dspec["new"].decode("latin-1")
        vold, vnew = vspec["old"].decode("latin-1"), vspec["new"].decode("latin-1")
        spec = [
            ("G0_original", 0, "none", "original", "none", None, None, None, "G0"),
            ("G1_resave", 0, t_main, "full", "none", None, None, None, "G1"),
            ("G2_recompress", 0, t_comp, "full_recompress", "none", None, None, None, "G2"),
            ("G3_note_incr", 0, "pymupdf", "incremental", "note", pg, None, None, "G3"),
            ("G4_linearize", 0, "pikepdf", "linearize", "none", None, None, None, "G4"),
            ("G5_stamp_honest", 0, "pymupdf", "full", "stamp", pg, None, None, "G5"),
            ("G6_textbox_honest", 0, "pymupdf", "incremental", "textbox", pg, None, None, "G6"),
            ("F1_digit_resave", 1, t_main, "full", "digit", pg, dold, dnew, "G1"),
            ("F2_digit_recompress", 1, t_comp, "full_recompress", "digit", pg, dold, dnew, "G2"),
            ("F3_digit_incr", 1, "pymupdf", "incremental", "digit", pg, dold, dnew, "G3"),
            ("F4_value_resave", 1, t_main, "full", "value", vspec["page"], vold, vnew, "G1"),
            ("F5_cover_replace", 1, "pymupdf", "full", "cover", cov["page"], cov["old"], cov["new"], "G1"),
            ("F6_digit_linearize", 1, "pikepdf", "linearize", "digit", pg, dold, dnew, "G4"),
            ("F7_stamp_fake", 1, "pymupdf", "full", "stamp", pg, None, None, "G5"),
            ("F8_textbox_fake", 1, "pymupdf", "incremental", "textbox", pg, None, None, "G6"),
        ]
        rows = [{"out_file": f"{stem}__{v}.pdf", "orig_file": name, "family": fam, "variant": v,
                 "label": lab, "tool": tool, "save_mode": mode, "edit_type": et, "edit_page": ep,
                 "old_value": ov, "new_value": nv, "twin": tw, "split": split}
                for v, lab, tool, mode, et, ep, ov, nv, tw in spec]
        with open(meta_path, "w") as fh:
            json.dump(rows, fh)
        return name, rows, None
    except Exception as ex:
        cleanup(stem)
        return name, None, f"{type(ex).__name__}: {str(ex)[:50]}"

# ---------- job list and chunk plan ----------
pool = pd.read_csv(CLEAN_POOL)
pool = pool[pool.producer_family.isin(FAMILIES)].copy()
pool["fam_order"] = pool.producer_family.map({f: i for i, f in enumerate(FAMILIES)})
pool = pool.sort_values(["fam_order", "file"])
JOBS = list(zip(pool.file, pool.producer_family))
CHUNK_LIST = [JOBS[i:i + CHUNK_SIZE] for i in range(0, len(JOBS), CHUNK_SIZE)]

print("Originals dir:", SRC, "|", len(glob.glob(SRC + "/*.pdf")), "PDFs")
print("\n=== ORIGINALS PER FAMILY (all clean files, zips 000-019) ===")
print(pool.producer_family.value_counts().reindex(FAMILIES).to_string())
print("Total originals:", len(JOBS), "| expected files if all succeed:", len(JOBS) * 15)
print("Dev share (originals):", round(sum(split_of(n) == "dev" for n, _ in JOBS) / len(JOBS) * 100, 1), "%")
print("\nChunks:", len(CHUNK_LIST), "of up to", CHUNK_SIZE, "originals")
print("Chunks already finished:", len(glob.glob(f"{CHUNKS}/chunk_*_features.csv")))
print("\nLibrary versions -> pikepdf", pikepdf.__version__, "| pymupdf", fitz.VersionBind, "| pypdf", pypdf.__version__)
free = shutil.disk_usage("/kaggle/working").free / 1e9
print(f"Free disk in /kaggle/working: {free:.1f} GB")

Originals dir: /kaggle/input/notebooks/atharvagandhi/pdp-bptp/govdocs_pdfs | 5029 PDFs

=== ORIGINALS PER FAMILY (all clean files, zips 000-019) ===
producer_family
acrobat distiller windows    1566
acrobat distiller for         659
acrobat pdfwriter for         428
adobe pdf library             153
corel pdf engine              108
Total originals: 2914 | expected files if all succeed: 43710
Dev share (originals): 14.8 %

Chunks: 10 of up to 300 originals
Chunks already finished: 0

Library versions -> pikepdf 10.18.0 | pymupdf 2.1 | pypdf 6.21.4
Free disk in /kaggle/working: 20.9 GB


In [2]:
# CELL 46 — TruFor (official code + weights, CVPR 2023) setup and smoke test. GPU check -> clone official repo -> download weights + MD5 check -> install yacs -> run the authors' test.py on (a) their 4 sample images, (b) rendered pages of 3 dev documents. Short output.
import os, sys, glob, json, shutil, subprocess, hashlib, time, zipfile
from collections import Counter
import numpy as np
import pandas as pd
import pymupdf as fitz
import torch

assert "process" in globals() and "JOBS" in globals() and "split_of" in globals(), "Run CELL 20 first"

def sh(cmd, timeout=600, cwd=None, env=None):
    """Run a shell command. Never raises. Returns (return code, combined output)."""
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, cwd=cwd, env=env)
        return r.returncode, (r.stdout or "") + (r.stderr or "")
    except subprocess.TimeoutExpired:
        return -9, f"TIMEOUT after {timeout}s"

def tail(s, n=8):
    return "\n".join(s.strip().splitlines()[-n:])

TF = "/kaggle/working/trufor"
REPO, TT = f"{TF}/TruFor", f"{TF}/TruFor/TruFor_train_test"
os.makedirs(TF, exist_ok=True)
ENV = dict(os.environ, TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD="1")   # newer PyTorch refuses the authors' checkpoint format without this
DPI = 150

# ---------- 1. GPU ----------
assert torch.cuda.is_available(), "No GPU - set Accelerator to GPU (T4 or P100) in Session options"
print(f"[✅] GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB | torch {torch.__version__}")

# ---------- 2. official code ----------
if not os.path.isdir(TT):
    rc, out = sh(f"git clone -q --depth 1 https://github.com/grip-unina/TruFor.git {REPO}", timeout=300)
    assert rc == 0, "git clone failed:\n" + tail(out)
print(f"[✅] TruFor code: {REPO}")

# ---------- 3. official weights + MD5 (from the authors' README) ----------
ZP, MD5 = f"{TF}/TruFor_weights.zip", "7bee48f3476c75616c3c5721ab256ff8"

def md5_of(p):
    h = hashlib.md5()
    with open(p, "rb") as fh:
        for ch in iter(lambda: fh.read(1 << 20), b""):
            h.update(ch)
    return h.hexdigest()

if not (os.path.exists(ZP) and md5_of(ZP) == MD5):
    rc, out = sh(f"curl -fL -sS -m 1200 -o {ZP} https://www.grip.unina.it/download/prog/TruFor/TruFor_weights.zip", timeout=1300)
    assert rc == 0 and os.path.exists(ZP), "weights download failed:\n" + tail(out)
assert md5_of(ZP) == MD5, "weights MD5 does not match the authors' value - do not continue, tell Claude"
with zipfile.ZipFile(ZP) as z:
    z.extractall(f"{TT}/pretrained_models")
hits = glob.glob(f"{TT}/pretrained_models/**/trufor.pth.tar", recursive=True)
assert hits, "trufor.pth.tar not found inside the weights zip"
WEIGHTS = hits[0]
print(f"[✅] Weights: MD5 matches the authors' value | {os.path.relpath(WEIGHTS, TT)} ({os.path.getsize(WEIGHTS) / 1e6:.0f} MB)")

# ---------- 4. packages ----------
rc, out = sh(f"{sys.executable} -m pip install -q yacs", timeout=300)
rc, out = sh(f'{sys.executable} -c "import yacs, timm, cv2, matplotlib; print(timm.__version__)"', timeout=120)
if rc != 0:
    sh(f"{sys.executable} -m pip install -q timm opencv-python-headless", timeout=600)
    rc, out = sh(f'{sys.executable} -c "import yacs, timm, cv2, matplotlib; print(timm.__version__)"', timeout=120)
assert rc == 0, "packages missing:\n" + tail(out)
print(f"[✅] Packages: yacs, timm {out.strip()}, opencv, matplotlib")

# ---------- runner: the authors' test.py (one call per folder); score = 'score' in each .npz ----------
def trufor(img_dir, out_dir, timeout=36000):
    os.makedirs(out_dir, exist_ok=True)
    t0 = time.time()
    rc, out = sh(f'{sys.executable} test.py -g 0 -in "{img_dir}" -out "{out_dir}" -exp trufor_ph3 TEST.MODEL_FILE "{WEIGHTS}"',
                 cwd=TT, timeout=timeout, env=ENV)
    sc = {}
    for img in sorted(glob.glob(f"{img_dir}/*")):
        npz = os.path.join(out_dir, os.path.basename(img) + ".npz")
        if os.path.exists(npz):
            sc[os.path.basename(img)] = float(np.load(npz)["score"])
    return rc, out, sc, time.time() - t0

# ---------- 5a. smoke test on the authors' own sample images ----------
SM = f"{TF}/smoke_authors"
shutil.rmtree(SM, ignore_errors=True)
shutil.copytree(f"{REPO}/test_docker/images", f"{SM}/img")
rc, out, sc, dt = trufor(f"{SM}/img", f"{SM}/out", timeout=900)
print(f"\n[{'✅' if len(sc) == 4 else '❌'}] Authors' sample images: {len(sc)} of 4 scored in {dt:.0f}s (score 0-1, higher = more likely edited)")
for k, v in sc.items():
    print(f"    {k:16s} {v:.3f}")
if len(sc) < 4:
    print(tail(out, 15))
ok_auth = len(sc) == 4 and max(v for k, v in sc.items() if "tampered" in k) > max(v for k, v in sc.items() if "pristine" in k)
print(f"[{'✅' if ok_auth else '⚠️'}] tampered samples score higher than pristine ones: {ok_auth}")

# ---------- 5b. smoke test on 3 dev documents (rendered pages where the edits are) ----------
dev_jobs = [j for j in JOBS if split_of(j[0]) == "dev"]
picks = [next(j for j in dev_jobs if j[1] == f) for f in ["acrobat distiller windows", "acrobat pdfwriter for", "corel pdf engine"]]
SD = f"{TF}/smoke_dev"
shutil.rmtree(SD, ignore_errors=True)
os.makedirs(f"{SD}/img")
n_img, sizes = 0, []
for job in picks:
    name, rows, err = process(job)
    assert rows is not None, f"generation failed for {name}: {err}"
    pages = sorted({int(r["edit_page"]) for r in rows if r["label"] == 1 and r["edit_page"] is not None})
    for r in rows:
        with fitz.open(os.path.join(OUT, r["out_file"])) as d:
            for pg in pages:
                if pg < len(d):
                    pix = d[pg].get_pixmap(dpi=DPI)
                    pix.save(f"{SD}/img/{r['out_file'][:-4]}__p{pg}.png")
                    sizes.append((pix.width, pix.height))
                    n_img += 1
    cleanup(name[:-4])
print(f"\nRendered {n_img} page images at {DPI} dpi | most common size (w, h): {Counter(sizes).most_common(1)[0][0] if sizes else None}")
rc, out, sc, dt = trufor(f"{SD}/img", f"{SD}/out", timeout=3600)
print(f"[{'✅' if len(sc) == n_img else '❌'}] TruFor scored {len(sc)} of {n_img} pages in {dt:.0f}s -> {dt / max(1, len(sc)):.2f}s per page")
if len(sc) < n_img:
    print("    last lines of TruFor output:\n" + tail(out, 15))
if sc:
    s = pd.Series(sc)
    tab = pd.DataFrame({"file": s.index.str.rsplit("__p", n=1).str[0], "score": s.values}).groupby("file").score.max()
    tab = tab.reset_index()
    tab["doc"] = tab.file.str.split("__").str[0]
    tab["variant"] = tab.file.str.split("__", n=1).str[1]
    print("\nTruFor score per file (max over the edited pages; higher = more likely edited):")
    print(tab.pivot(index="variant", columns="doc", values="score").round(3).to_string())
json.dump({"gpu": torch.cuda.get_device_name(0), "weights": WEIGHTS, "dpi": DPI, "authors_ok": ok_auth,
           "dev_pages": n_img, "dev_scored": len(sc), "sec_per_page": round(dt / max(1, len(sc)), 3)},
          open(f"{TF}/cell46_report.json", "w"), indent=2)
print(f"\n=== CELL 46 DONE === report: {TF}/cell46_report.json")

[✅] GPU: Tesla T4 | 16 GB | torch 2.11.0+cu130
[✅] TruFor code: /kaggle/working/trufor/TruFor
[✅] Weights: MD5 matches the authors' value | pretrained_models/weights/trufor.pth.tar (281 MB)
[✅] Packages: yacs, timm 1.0.30, opencv, matplotlib

[✅] Authors' sample images: 4 of 4 scored in 18s (score 0-1, higher = more likely edited)
    pristine1.jpg    0.190
    pristine2.jpg    0.188
    tampered1.png    0.998
    tampered2.png    0.998
[✅] tampered samples score higher than pristine ones: True

Rendered 90 page images at 150 dpi | most common size (w, h): (1275, 1650)
[✅] TruFor scored 90 of 90 pages in 300s -> 3.33s per page

TruFor score per file (max over the edited pages; higher = more likely edited):
doc                  000010  000263  003178
variant                                    
F1_digit_resave       0.121   0.282   0.204
F2_digit_recompress   0.121   0.282   0.204
F3_digit_incr         0.121   0.282   0.204
F4_value_resave       0.121   0.284   0.203
F5_cover_replace    

In [ ]:
# CELL 47 — TruFor on ALL 403 dev documents (same files as tables B1-B4). Per chunk of 20 documents: regenerate the 15 versions (CELL 20 generator) -> render the edited pages (150 dpi) of every version -> authors' test.py on all GPUs in parallel -> save page scores -> delete images. Resumable; stops starting new chunks after 11 h. Then: file score = max over its edited pages, and the same metrics as B1-B4 (CELL 23 functions).
import os, sys, glob, json, time, shutil, subprocess
import numpy as np
import pandas as pd
import pymupdf as fitz
import torch
from concurrent.futures import ThreadPoolExecutor, TimeoutError, as_completed
from pebble import ProcessPool
from sklearn.metrics import roc_auc_score, average_precision_score

assert "process" in globals() and "JOBS" in globals(), "Run CELL 20 first"
assert "WEIGHTS" in globals() and "TT" in globals() and "ENV" in globals(), "Run CELL 46 first"
T_START = time.time()
MAX_HOURS = 11.0                                     # Kaggle limit is 12 h; leave time to compute and save results
N_GPU = max(1, torch.cuda.device_count())
FULLD = f"{TF}/full"
CHK = f"{FULLD}/chunks"
os.makedirs(CHK, exist_ok=True)

# ---------- dev documents and their edited pages (from full_features.csv) ----------
hf = sorted(glob.glob("/kaggle/input/**/full/full_features.csv", recursive=True))
assert hf, "full_features.csv not found - is BTP Full attached as input?"
REF = pd.read_csv(hf[0], usecols=["out_file", "orig_file", "family", "variant", "label", "split", "edit_page", "tool"])
DEV = REF[REF.split == "dev"].reset_index(drop=True)
PAGES = DEV[DEV.label == 1].dropna(subset=["edit_page"]).groupby("orig_file").edit_page.apply(lambda s: sorted({int(x) for x in s})).to_dict()
fam_of = dict(zip(DEV.orig_file, DEV.family))
by_fam = {}
for o in sorted(DEV.orig_file.unique()):
    by_fam.setdefault(fam_of[o], []).append(o)
order = []                                           # round-robin over families -> every chunk mixes families
while any(by_fam.values()):
    for f in sorted(by_fam):
        if by_fam[f]:
            order.append(by_fam[f].pop(0))
job_of = {j[0]: j for j in JOBS}
assert all(o in job_of for o in order), "some dev documents are not in CELL 20's job list"
CHUNKS_T = [order[i:i + 20] for i in range(0, len(order), 20)]
print(f"Dev documents: {len(order)} | dev files: {len(DEV)} | chunks of 20: {len(CHUNKS_T)} | GPUs: {N_GPU} ({torch.cuda.get_device_name(0)})")

def run_tf(img_dir, out_dir, gpu):                   # the authors' test.py on one folder, on one GPU
    os.makedirs(out_dir, exist_ok=True)
    return sh(f'{sys.executable} test.py -g {gpu} -in "{img_dir}" -out "{out_dir}" -exp trufor_ph3 TEST.MODEL_FILE "{WEIGHTS}"',
              cwd=TT, timeout=6 * 3600, env=ENV)

def gen_pool(jobs):
    res, err = {}, {}
    with ProcessPool(max_workers=4) as pex:
        futs = {pex.schedule(process, args=(j,), timeout=TIME_LIMIT): j for j in jobs}
        for fu in as_completed(futs):
            j = futs[fu]
            try:
                res[j] = fu.result()
            except Exception as ex:
                err[j] = f"{type(ex).__name__}: {str(ex)[:60]}"
    return res, err

def run_chunk(ci):
    fcsv = f"{CHK}/chunk_{ci:02d}.csv"
    if os.path.exists(fcsv):
        return "skipped"
    t0 = time.time()
    work = f"{FULLD}/work"
    shutil.rmtree(work, ignore_errors=True)
    for p in glob.glob(os.path.join(OUT, "*.pdf")):
        os.remove(p)
    docs = CHUNKS_T[ci]
    res, err = gen_pool([job_of[o] for o in docs])
    imgs, fails = [], []
    for o in docs:
        r = res.get(job_of[o])
        if r is None or r[1] is None:
            fails.append(o)
            continue
        want = set(DEV.loc[DEV.orig_file == o, "out_file"])
        for row in r[1]:
            if row["out_file"] not in want:
                continue
            with fitz.open(os.path.join(OUT, row["out_file"])) as d:
                for pg in PAGES.get(o, []):
                    if pg < len(d):
                        imgs.append((row["out_file"], pg))
                        g = len(imgs) % N_GPU                # spread images over the GPUs
                        os.makedirs(f"{work}/img{g}", exist_ok=True)
                        d[pg].get_pixmap(dpi=DPI).save(f"{work}/img{g}/{row['out_file'][:-4]}__p{pg}.png")
        cleanup(o[:-4])
    with ThreadPoolExecutor(N_GPU) as ex:
        outs = list(ex.map(lambda g: run_tf(f"{work}/img{g}", f"{work}/out{g}", g),
                           [g for g in range(N_GPU) if os.path.isdir(f"{work}/img{g}")]))
    rows = []
    for f, pg in imgs:
        npz = glob.glob(f"{work}/out*/{f[:-4]}__p{pg}.png.npz")
        rows.append({"out_file": f, "page": pg, "score": float(np.load(npz[0])["score"]) if npz else np.nan})
    df = pd.DataFrame(rows, columns=["out_file", "page", "score"])
    df.to_csv(fcsv + ".tmp", index=False)
    os.replace(fcsv + ".tmp", fcsv)
    shutil.rmtree(work, ignore_errors=True)
    for p in glob.glob(os.path.join(OUT, "*.pdf")):
        os.remove(p)
    n_miss = int(df.score.isna().sum())
    msg = (f"Chunk {ci + 1}/{len(CHUNKS_T)}: docs {len(docs) - len(fails)}/{len(docs)} | pages {len(df)} | not scored {n_miss}"
           f" | {(time.time() - t0) / 60:.1f} min | elapsed {(time.time() - T_START) / 3600:.2f} h")
    if n_miss:
        msg += "\n    TruFor output (last lines): " + " | ".join(tail(o[1], 3).replace("\n", " ") for o in outs)
    if fails:
        msg += f"\n    generation failed: {fails} {[err.get(job_of[o], '') for o in fails]}"
    return msg

for ci in range(len(CHUNKS_T)):
    if (time.time() - T_START) / 3600 > MAX_HOURS:
        print(f"Time limit reached ({MAX_HOURS} h) - stopping before chunk {ci + 1}; results below use the finished chunks")
        break
    print(run_chunk(ci), flush=True)

# ---------- results: file score = max over its edited pages ----------
pages = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(f"{CHK}/chunk_*.csv"))], ignore_index=True)
pages.to_csv(f"{FULLD}/trufor_page_scores.csv", index=False)
fs = pages.dropna(subset=["score"]).groupby("out_file").score.max().rename("trufor_score").reset_index()
te = DEV.merge(fs, on="out_file", how="inner")
full_docs = te.groupby("orig_file").out_file.count()
te = te[te.orig_file.isin(full_docs[full_docs == 15].index)].reset_index(drop=True)   # documents with all 15 versions scored
te.to_csv(f"{FULLD}/trufor_dev_scores.csv", index=False)
print(f"\nScored: {te.orig_file.nunique()} of {len(order)} dev documents ({len(te)} files) | pages scored {int(pages.score.notna().sum())} of {len(pages)}")

NO_EV = ["F2_digit_recompress", "F7_stamp_fake", "F8_textbox_fake", "G2_recompress", "G5_stamp_honest", "G6_textbox_honest"]
PAIRS = [("G1_resave", "F1_digit_resave"), ("G2_recompress", "F2_digit_recompress"), ("G3_note_incr", "F3_digit_incr"),
         ("G1_resave", "F4_value_resave"), ("G1_resave", "F5_cover_replace"), ("G4_linearize", "F6_digit_linearize"),
         ("G5_stamp_honest", "F7_stamp_fake"), ("G6_textbox_honest", "F8_textbox_fake")]

def tpr_at(y, s, fpr):                               # CELL 23
    thr = np.quantile(s[y == 0], 1 - fpr)
    return float((s[y == 1] > thr).mean())

def auc_or_nan(y, s):                                # CELL 23
    try:
        return round(roc_auc_score(y, s), 3)
    except ValueError:
        return np.nan

def scores(y, s):                                    # CELL 23
    y, s = np.asarray(y), np.asarray(s, dtype=float)
    return {"ROC-AUC": auc_or_nan(y, s), "PR-AUC": round(average_precision_score(y, s), 3),
            "Det@1%FA": round(tpr_at(y, s, 0.01), 3), "Det@5%FA": round(tpr_at(y, s, 0.05), 3)}

EVID = ~te.variant.isin(NO_EV)
s = te.trufor_score.values
r_all = {"Method": "TruFor (CVPR 2023, official weights; max over edited pages)", **scores(te.label, s)}
r_ev = {"Method": r_all["Method"], **scores(te.label[EVID], s[EVID.values])}
r_tw = {"Method": r_all["Method"]}
for g, f in PAIRS:
    m = te.variant.isin([g, f]).values
    r_tw[f.split("_")[0] + " vs " + g.split("_")[0]] = auc_or_nan(te.label[m], s[m])
rng = np.random.default_rng(0)
docs = te.orig_file.unique()
by_doc = {d: te.index[te.orig_file == d].values for d in docs}
ba, be = [], []
for _ in range(300):
    idx = np.concatenate([by_doc[d] for d in rng.choice(docs, size=len(docs), replace=True)])
    y, ev = te.label.values[idx], EVID.values[idx]
    ba.append(roc_auc_score(y, s[idx]))
    be.append(roc_auc_score(y[ev], s[idx][ev]))
r_ci = {"Method": r_all["Method"],
        "AUC all [95% CI]": f"{np.mean(ba):.3f} [{np.percentile(ba, 2.5):.3f}, {np.percentile(ba, 97.5):.3f}]",
        "AUC evidence-bearing [95% CI]": f"{np.mean(be):.3f} [{np.percentile(be, 2.5):.3f}, {np.percentile(be, 97.5):.3f}]"}
pd.set_option("display.width", 250)
out_txt = []
for title, row in [("T1: ALL 15 VERSIONS", r_all), ("T2: EVIDENCE-BEARING ONLY", r_ev), ("T3: TWIN AUC (0.5 = cannot tell them apart)", r_tw),
                   ("T4: 95% CONFIDENCE INTERVALS (bootstrap over dev documents)", r_ci)]:
    t = pd.DataFrame([row]).to_string(index=False)
    print(f"\n=== {title} ===\n{t}")
    out_txt.append(f"=== {title} ===\n{t}\n")
spread = te.groupby("orig_file").trufor_score.agg(lambda x: x.max() - x.min())
print(f"\nPer document: max - min TruFor score over its 15 versions -> median {spread.median():.3f} | 90th percentile {spread.quantile(0.9):.3f}")
with open(f"{FULLD}/TRUFOR_RESULTS.txt", "w") as fh:
    fh.write(f"Dev documents scored: {te.orig_file.nunique()} of {len(order)} | files {len(te)} | DPI {DPI} | GPUs {N_GPU}\n\n" + "\n".join(out_txt))
    fh.write(f"\nPer-document score spread over 15 versions: median {spread.median():.3f}, 90th pct {spread.quantile(0.9):.3f}\n")
print(f"\nSaved: {FULLD}/trufor_page_scores.csv, trufor_dev_scores.csv, TRUFOR_RESULTS.txt | total time {(time.time() - T_START) / 3600:.2f} h")